In [ ]:
# Install the Telethon library for Telegram API interactions
!pip install -q telethon requests beautifulsoup4

from datetime import datetime, timezone
import pandas as pd
import time
import json
import re


from telethon.sync import TelegramClient

from google.colab import files



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.9/715.9 kB 7.4 MB/s eta 0:00:00


In [ ]:
api_id = 'api_id'
api_hash = 'hash'
username = 'name'
phone="+number"


In [ ]:
channels = "ctinow"
channels = [channel.strip() for channel in channels.split(",")]

date_min = '2023-01-01'
date_max = '2025-03-01'

date_min = datetime.fromisoformat(date_min).replace(tzinfo=timezone.utc)
date_max = datetime.fromisoformat(date_max).replace(tzinfo=timezone.utc)

file_name = 'ctinow'

key_search = ''

max_t_index = 100000

time_limit = 21600

File = 'excel'



In [ ]:
data = []
t_index = 0
start_time = time.time()


In [ ]:

def format_time(seconds):
    days = seconds // 86400
    hours = (seconds % 86400) // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    return f'{int(days):02}:{int(hours):02}:{int(minutes):02}:{int(seconds):02}'


In [ ]:
# Function to print progress of the scraping process and estimate remaining time
def print_progress(t_index, message_id, start_time, max_t_index):
    elapsed_time = time.time() - start_time
    current_progress = t_index / (t_index + message_id) if (t_index + message_id) <= max_t_index else t_index / max_t_index
    percentage = current_progress * 100
    estimated_total_time = elapsed_time / current_progress
    remaining_time = estimated_total_time - elapsed_time

    elapsed_time_str = format_time(elapsed_time)
    remaining_time_str = format_time(remaining_time)

    print(f'Progress: {percentage:.2f}% | Elapsed Time: {elapsed_time_str} | Remaining Time: {remaining_time_str}')



In [ ]:
# Normalize File variable to avoid issues
File = re.sub(r'[^a-z]', '', File.lower())  # Converts to lowercase and removes non-alphabetic characters


In [ ]:
def extract_links(text):

    url_pattern = r'https?://[^\s,]+|www\.[^\s,]+\.\w+'
    return re.findall(url_pattern, str(text))

In [ ]:

def clean_text(text):


    text = re.sub(r'https?://\S+|www\.\S+', '', str(text))

     # Remove emojis
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)


    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Start scraping process
for channel in channels:
    if t_index >= max_t_index:
        break

    if time.time() - start_time > time_limit:
        break

    loop_start_time = time.time()

    try:
        c_index = 0
        async with TelegramClient(username, api_id, api_hash) as client:
            async for message in client.iter_messages(channel, search=key_search):
                try:
                    if date_min <= message.date <= date_max:
                        # Process comments of the message
                        comments_list = []
                        try:
                            async for comment_message in client.iter_messages(channel, reply_to=message.id):
                                comment_text = clean_text(comment_message.text)
                                comment_media = 'True' if comment_message.media else 'False'

                                comment_emoji_string = ''
                                if comment_message.reactions:
                                    for reaction_count in comment_message.reactions.results:
                                        emoji = reaction_count.reaction.emoticon
                                        count = str(reaction_count.count)
                                        comment_emoji_string += f"{emoji} {count} "

                                comment_date_time = comment_message.date.strftime('%Y-%m-%d %H:%M:%S')

                                comments_list.append({
                                    'Type': 'comment',
                                    'Comment Group': channel,
                                    'Comment Author ID': comment_message.sender_id,
                                    'Comment Content': comment_text,
                                    'Comment Date': comment_date_time,
                                    'Comment Message ID': comment_message.id,
                                    'Comment Author': comment_message.post_author,
                                    'Comment Views': comment_message.views,
                                    'Comment Reactions': comment_emoji_string,
                                    'Comment Shares': comment_message.forwards,
                                    'Comment Media': comment_media,
                                    'Comment Url': f'https://t.me/{channel}/{message.id}?comment={comment_message.id}'.replace('@', ''),
                                })
                        except Exception as e:
                            comments_list = []
                            print(f'Error processing comments: {e}')

                        # Process the main message
                        cleaned_content = clean_text(message.text)
                        message_links = extract_links(message.text)

                        media = 'True' if message.media else 'False'

                        emoji_string = ''
                        if message.reactions:
                            for reaction_count in message.reactions.results:
                                emoji = reaction_count.reaction.emoticon
                                count = str(reaction_count.count)
                                emoji_string += f"{emoji} {count} "

                        date_time = message.date.strftime('%Y-%m-%d %H:%M:%S')
                        cleaned_comments_list = clean_text(json.dumps(comments_list))

                        data.append({
                            'Type': 'text',
                            'Group': channel,
                            'Author ID': message.sender_id,
                            'Content': cleaned_content,  # Full text message captured
                            'Message Links': json.dumps(message_links),
                            'Date':date_time,
                            'Message ID': message.id,
                            'Author': message.post_author,
                            'Views': message.views,
                            'Reactions': emoji_string,
                            'Shares': message.forwards,
                            'Media': media,
                            'Url': f'https://t.me/{channel}/{message.id}'.replace('@', ''),
                            'Comments List': cleaned_comments_list,
                        })

                        c_index += 1
                        t_index += 1

                        # Print progress
                        print(f'{"-" * 80}')
                        print(f'From {channel}: {c_index:05} contents of {t_index:05}')
                        print(f'Id: {message.id:05} / Date: {date_time}')
                        print(f'Total: {t_index:05} contents until now')
                        print(f'{"-" * 80}\n\n')

                        # Save backup every 1000 messages
                        if t_index % 1000 == 0:
                            backup_filename = f'backup_{file_name}_until_{t_index:05}_{channel}_ID{message.id:07}.parquet'
                            pd.DataFrame(data).to_parquet(backup_filename, index=False)

                        # Exit conditions
                        if t_index >= max_t_index:
                            break
                        if time.time() - start_time > time_limit:
                            break

                    elif message.date < date_min:
                        break

                except Exception as e:
                    print(f'Error processing message: {e}')
                    # Save channel-specific file
        print(f'\n\n##### {channel} was ok with {c_index:05} posts #####\n\n')
        df = pd.DataFrame(data)
        partial_filename = f'complete_{channel}_in_{file_name}_until_{t_index:05}.parquet'
        df.to_parquet(partial_filename, index=False)

    except Exception as e:
        print(f'{channel} error: {e}')

    # Respect Telegram's rate limits
    loop_end_time = time.time()
    loop_duration = loop_end_time - loop_start_time
    if loop_duration < 60:
        time.sleep(60 - loop_duration)


In [ ]:
print(f'\n{"-" * 50}\n#Concluded! #{t_index:05} posts were scraped!\n{"-" * 50}\n\n\n\n')
df = pd.DataFrame(data)
if File == 'parquet':
    final_filename = f'{file_name}__{t_index:05}.parquet'
    df.to_parquet(final_filename, index=False)
elif File == 'excel':
    final_filename = f'{file_name}__{t_index:05}.xlsx'
    df.to_excel(final_filename, index=False, engine='openpyxl')
files.download(final_filename)




--------------------------------------------------
#Concluded! #30927 posts were scraped!
--------------------------------------------------






<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>